# Week 3 — Data Preprocessing

## 1. Missing-Data Assessment

Analyze missing pollutant readings in Delhi before selecting appropriate preprocessing methods.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/raw/city_day.csv")

delhi_df = df[df["City"] == "Delhi"].copy()
delhi_df["Date"] = pd.to_datetime(delhi_df["Date"])

delhi_df = delhi_df.sort_values("Date")
delhi_df = delhi_df.set_index("Date").asfreq("D")

print("Delhi dataset shape:", delhi_df.shape)

Delhi dataset shape: (2009, 15)


In [2]:
missing_summary = pd.DataFrame({
    "Missing Count": delhi_df.isna().sum(),
    "Missing %": (delhi_df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary.sort_values(
    "Missing %", ascending=False
)

display(missing_summary)

,Missing Count,Missing %
Xylene,781,38.88
SO2,110,5.48
O3,84,4.18
PM10,77,3.83
AQI,10,0.50
AQI_Bucket,10,0.50
NH3,9,0.45
PM2.5,2,0.10
NO,2,0.10
NO2,2,0.10


## 2. Missing-Value Gap Analysis

Examine consecutive missing-value periods to determine whether forward filling is appropriate.

In [3]:
def longest_missing_gap(series):
    missing = series.isna()
    groups = (~missing).cumsum()
    return missing.groupby(groups).sum().max()

gap_summary = pd.DataFrame({
    "Missing Count": delhi_df.isna().sum(),
    "Longest Missing Gap": [
        longest_missing_gap(delhi_df[col])
        for col in delhi_df.columns
    ]
})

display(gap_summary.sort_values("Longest Missing Gap", ascending=False))

,Missing Count,Longest Missing Gap
Xylene,781,606
SO2,110,81
O3,84,81
PM10,77,58
AQI,10,4
AQI_Bucket,10,4
PM2.5,2,2
NO,2,2
NO2,2,2
NH3,9,2


In [4]:
print("Dates with missing PM2.5:")
display(delhi_df[delhi_df["PM2.5"].isna()][["PM2.5", "AQI", "AQI_Bucket"]])

Dates with missing PM2.5:


,PM2.5,AQI,AQI_Bucket
Date,,,
2017-08-12,NaN,45.0,Good
2017-08-13,NaN,NaN,NaN


## 3. Proposed Missing-Data Treatment Policy

- Exclude Xylene due to high missingness (38.88%) and a maximum consecutive gap of 606 days.
- Forward-fill short gaps (maximum 2 consecutive days) in selected pollutant predictors.
- Preserve longer missing periods instead of fabricating extended observations.
- Preserve missing PM2.5 and AQI_Bucket values because these columns are used to construct prediction targets.
- Use only previously observed values for filling to prevent look-ahead leakage.

**Status:** Standard preprocessing policy adopted for Week 3.

All subsequent feature-engineering tasks will use this preprocessing pipeline to maintain consistency across the project.

In [5]:
pollutant_cols = ["SO2", "O3", "PM10", "NO", "NO2", "NH3"]

for col in pollutant_cols:
    missing = delhi_df[col].isna()
    gap_id = missing.ne(missing.shift()).cumsum()
    gap_length = missing.groupby(gap_id).transform("sum")

    eligible = missing & (gap_length <= 2)

    print(f"{col}: {eligible.sum()} missing days in short gaps")

SO2: 1 missing days in short gaps
O3: 3 missing days in short gaps
PM10: 9 missing days in short gaps
NO: 2 missing days in short gaps
NO2: 2 missing days in short gaps
NH3: 9 missing days in short gaps


## 4. Apply Missing-Data Treatment

Create a separate processed DataFrame, exclude Xylene, and forward-fill only short missing gaps in selected pollutant predictors.

In [6]:
processed_df = delhi_df.drop(columns=["Xylene"]).copy()

for col in pollutant_cols:
    missing = processed_df[col].isna()

    gap_id = missing.ne(missing.shift()).cumsum()
    gap_length = missing.groupby(gap_id).transform("sum")

    eligible = missing & (gap_length <= 2)

    # Only use previously observed values
    previous_value = processed_df[col].ffill()

    # Fill only eligible gaps
    processed_df.loc[eligible, col] = previous_value[eligible]

print("Processed dataset shape:", processed_df.shape)

Processed dataset shape: (2009, 14)


In [7]:
comparison = pd.DataFrame({
    "Before": delhi_df.isna().sum(),
    "After": processed_df.isna().sum()
})

comparison["Filled"] = comparison["Before"] - comparison["After"]

display(comparison)

,Before,After,Filled
AQI,10,10.0,0.0
AQI_Bucket,10,10.0,0.0
Benzene,0,0.0,0.0
CO,0,0.0,0.0
City,0,0.0,0.0
NH3,9,0.0,9.0
NO,2,0.0,2.0
NO2,2,0.0,2.0
NOx,0,0.0,0.0
O3,84,81.0,3.0


## 5. Preprocessing Validation

Verify that original non-missing measurements remain unchanged and identify remaining missing values.

In [8]:
# Verify original non-missing values remain unchanged
original_values = delhi_df[processed_df.columns]

unchanged = (
    processed_df.where(original_values.notna())
    .equals(original_values.where(original_values.notna()))
)

print("Original valid measurements preserved:", unchanged)

print("PM2.5 unchanged:",
      processed_df["PM2.5"].equals(delhi_df["PM2.5"]))

print("AQI_Bucket unchanged:",
      processed_df["AQI_Bucket"].equals(delhi_df["AQI_Bucket"]))

Original valid measurements preserved: True
PM2.5 unchanged: True
AQI_Bucket unchanged: True


In [9]:
remaining_missing = processed_df.isna().sum()

print("Remaining missing values:")
print(remaining_missing[remaining_missing > 0])

print("\nTotal remaining missing values:",
      remaining_missing.sum())

Remaining missing values:
PM2.5           2
PM10           68
SO2           109
O3             81
AQI            10
AQI_Bucket     10
dtype: int64

Total remaining missing values: 280


## 6. Preprocessing Summary

- Excluded Xylene due to 38.88% missingness and a maximum missing gap of 606 days.
- Filled 26 missing pollutant readings belonging to gaps of at most two consecutive days, using previously available measurements.
- Preserved longer missing periods in PM10, SO2, and O3.
- Preserved missing PM2.5, AQI, and AQI_Bucket values.
- Verified that all original non-missing measurements remain unchanged.
- The processed dataset contains 2,009 rows and 14 columns, with 280 missing values remaining.

**Status:** Standard preprocessing policy adopted for Week 3.

All subsequent feature-engineering tasks will use this preprocessing pipeline to maintain consistency across the project.

**Limitation:** Remaining missing predictor values must be handled consistently during model preparation. Forward filling does not use future readings, but further validation is required to ensure no train/test leakage.

## 7. Pollutant Lag Features — Deepanshu

Lag features use historical pollutant measurements to help predict next-day air quality. For each selected pollutant, we create lag features for 1, 2, 3, and 7 days before the current date. The dataset is maintained on a continuous daily calendar so that each lag corresponds to the correct number of days. Missing values are preserved rather than filled using future observations.

In [10]:
# Pollutant lag features — Deepanshu

processed_df = processed_df.sort_index().copy()

pollutants_for_lags = [
    "PM2.5", "PM10", "NO", "NO2", "NOx",
    "NH3", "CO", "SO2", "O3", "Benzene", "Toluene"
]

# Keep only pollutant columns that exist in the dataset
pollutants_for_lags = [
    col for col in pollutants_for_lags
    if col in processed_df.columns
]

lag_periods = [1, 2, 3, 7]

for pollutant in pollutants_for_lags:
    for lag in lag_periods:
        processed_df[f"{pollutant}_lag_{lag}"] = (
            processed_df[pollutant].shift(lag)
        )

print("Lag features created successfully.")
print("Dataset shape:", processed_df.shape)

lag_columns = [
    f"{pollutant}_lag_{lag}"
    for pollutant in pollutants_for_lags
    for lag in lag_periods
]

display(processed_df[lag_columns].head(10))

Lag features created successfully.
Dataset shape: (2009, 58)


,PM2.5_lag_1,PM2.5_lag_2,PM2.5_lag_3,PM2.5_lag_7,PM10_lag_1,PM10_lag_2,PM10_lag_3,PM10_lag_7,NO_lag_1,NO_lag_2,...,O3_lag_3,O3_lag_7,Benzene_lag_1,Benzene_lag_2,Benzene_lag_3,Benzene_lag_7,Toluene_lag_1,Toluene_lag_2,Toluene_lag_3,Toluene_lag_7
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-02,313.22,NaN,NaN,NaN,607.98,NaN,NaN,NaN,69.16,NaN,...,NaN,NaN,14.36,NaN,NaN,NaN,24.86,NaN,NaN,NaN
2015-01-03,186.18,313.22,NaN,NaN,269.55,607.98,NaN,NaN,62.09,69.16,...,NaN,NaN,10.55,14.36,NaN,NaN,20.09,24.86,NaN,NaN
2015-01-04,87.18,186.18,313.22,NaN,131.90,269.55,607.98,NaN,25.73,62.09,...,41.68,NaN,3.91,10.55,14.36,NaN,10.23,20.09,24.86,NaN
2015-01-05,151.84,87.18,186.18,NaN,241.84,131.90,269.55,NaN,25.01,25.73,...,29.97,NaN,4.26,3.91,10.55,NaN,9.71,10.23,20.09,NaN
2015-01-06,146.60,151.84,87.18,NaN,219.13,241.84,131.90,NaN,14.01,25.01,...,19.71,NaN,2.80,4.26,3.91,NaN,6.21,9.71,10.23,NaN
2015-01-07,149.58,146.60,151.84,NaN,252.10,219.13,241.84,NaN,17.21,14.01,...,25.36,NaN,3.63,2.80,4.26,NaN,7.35,6.21,9.71,NaN
2015-01-08,217.87,149.58,146.60,313.22,376.51,252.10,219.13,607.98,26.99,17.21,...,23.20,41.68,4.93,3.63,2.80,14.36,9.42,7.35,6.21,24.86
2015-01-09,229.90,217.87,149.58,186.18,360.95,376.51,252.10,269.55,23.34,26.99,...,26.83,29.97,5.80,4.93,3.63,10.55,11.40,9.42,7.35,20.09


In [11]:
# Verify lag direction on one date

check_date = pd.Timestamp("2017-01-10")

if check_date in processed_df.index:
    check = processed_df.loc[check_date]

    print("Date:", check_date.date())
    print("PM2.5 on current date:", check["PM2.5"])
    print("PM2.5 lag 1 (previous day):", check["PM2.5_lag_1"])
    print("PM2.5 lag 2:", check["PM2.5_lag_2"])
    print("PM2.5 lag 3:", check["PM2.5_lag_3"])
    print("PM2.5 lag 7:", check["PM2.5_lag_7"])
else:
    print("Check date not found.")

Date: 2017-01-10
PM2.5 on current date: 108.26
PM2.5 lag 1 (previous day): 141.88
PM2.5 lag 2: 136.98
PM2.5 lag 3: 154.52
PM2.5 lag 7: 255.33


In [12]:
print("Index type:", type(processed_df.index).__name__)
print("Index sorted:", processed_df.index.is_monotonic_increasing)
print("Duplicate dates:", processed_df.index.duplicated().sum())

date_gaps = processed_df.index.to_series().diff().dropna()
print("Non-daily intervals:", (date_gaps != pd.Timedelta(days=1)).sum())

print("Number of lag columns:", len(lag_columns))
print("Missing values in lag columns:", processed_df[lag_columns].isna().sum().sum())

Index type: DatetimeIndex
Index sorted: True
Duplicate dates: 0
Non-daily intervals: 0
Number of lag columns: 44
Missing values in lag columns: 1183
